# SaaS Revenue & Churn Analysis

## Business Objective
Measure recurring revenue, customer churn, and revenue exposure to churn, then translate the results into retention priorities.

## 1. Load the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/saas_customers.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['start_date', 'churn_date'])
df['status'] = df['churn_date'].notna().map({True: 'Churned', False: 'Active'})
df.head()

## 2. Data quality checks

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate customer IDs:', df['customer_id'].duplicated().sum())
print('\nStatus distribution:\n', df['status'].value_counts())
assert df['customer_id'].is_unique
assert df['monthly_fee'].ge(0).all()

## 3. Key business KPIs

In [ ]:
active = df[df.status == 'Active']
churned = df[df.status == 'Churned']
mrr = active.monthly_fee.sum()
arr = mrr * 12
churn_rate = len(churned) / len(df) * 100
revenue_at_risk = churned.monthly_fee.sum()
pd.Series({'Customers': len(df), 'Active customers': len(active), 'Churned customers': len(churned), 'MRR ($)': mrr, 'ARR ($)': arr, 'Churn rate (%)': churn_rate, 'Monthly revenue at risk ($)': revenue_at_risk})

## 4. Revenue and churn by plan

In [ ]:
plan = df.groupby('plan').agg(customers=('customer_id','count'), churned=('status', lambda x: (x=='Churned').sum()), active_mrr=('monthly_fee', lambda x: x[df.loc[x.index,'status'].eq('Active')].sum()))
plan['churn_rate_pct'] = plan['churned'] / plan['customers'] * 100
plan.sort_values('active_mrr', ascending=False)

In [ ]:
plan['active_mrr'].sort_values(ascending=False).plot(kind='bar', figsize=(8,5), title='Active MRR by Plan'); plt.ylabel('MRR ($)'); plt.xlabel(''); plt.tight_layout(); plt.show()
plan['churn_rate_pct'].sort_values(ascending=False).plot(kind='bar', figsize=(8,5), title='Customer Churn Rate by Plan'); plt.ylabel('Churn Rate (%)'); plt.xlabel(''); plt.tight_layout(); plt.show()

## 5. Customer segment analysis

In [ ]:
segment = df.groupby('segment').agg(customers=('customer_id','count'), churned=('status', lambda x: (x=='Churned').sum()), active_mrr=('monthly_fee', lambda x: x[df.loc[x.index,'status'].eq('Active')].sum()))
segment['churn_rate_pct'] = segment['churned'] / segment['customers'] * 100
segment.sort_values('active_mrr', ascending=False)

In [ ]:
segment['active_mrr'].sort_values(ascending=False).plot(kind='bar', figsize=(8,5), title='Active MRR by Customer Segment'); plt.ylabel('MRR ($)'); plt.xlabel(''); plt.tight_layout(); plt.show()

## 6. Revenue at risk

In [ ]:
revenue_risk = churned.groupby('plan')['monthly_fee'].sum().sort_values(ascending=False)
revenue_risk

## 7. Key insights and recommendations

- Starter has the highest plan churn rate in the verified dataset at 12.9%.
- Growth has 10.0% churn, making it the second-highest churn plan.
- Enterprise has 0% churn in this sample and contributes $5,489 of active MRR.
- SMB has the highest churn rate at 13.3% and is the largest customer segment.
- Monthly recurring revenue is $14,858, equivalent to $178,296 ARR.
- Monthly revenue at risk from churned customers is $692.
- Recommended actions: strengthen onboarding for Starter and Growth customers, monitor SMB accounts for early churn signals, and protect high-MRR Enterprise accounts with proactive retention outreach.